# Hyperparameter sensitivity — 04 connectivity methods

One curve per swept hyperparameter, for each of the five arms: 16-task
non-stationary multi-MNIST, 800,000 steps, 5 vmapped seeds per cell.

**Each point is that parameter value at its own best setting of everything
else** — not an average over the rest of the grid. Marginalising would punish a
value that works only in a narrow corner, which is the opposite of what a
sensitivity curve is for: the question is "how good can this value be made",
not "how good is it on average". Concretely, for a point at
`evolve_frequency=250` we take the minimum asymptotic loss over every
(width, learning rate) pair at that frequency.

Lower is better throughout — the y axis is loss.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from multi_mnist.analysis import (
    asymptotic_metric, filter_sweep, load_export, normalize_columns,
    plot_sensitivity, save_fig, set_style,
)

%matplotlib inline
set_style()

## Sweeps, axes and cells

Each arm spans one or two Comet sweep names: the v2 grid, plus a `v2b` top-up
that re-ran one width after the original sweep lost its remaining assignments.
`filter_sweep` merges them and `dedupe_trials` keeps one trial per cell,
preferring trials that logged every expected period (a divergence counts as
finished, since stopping early on divergence is a completed measurement).
That is what collapses both the top-up overlap and the failed retries.

In [ ]:
PROJECT = 'paper-weight-pruning-connectivity-sweep'

SWEEPS = {
    'SET':           ['04_set_n16_v2', '04_set_n16_v2b_h1024'],
    'DEEP-R':        ['04_deep_r_n16_v2', '04_deep_r_n16_v2b_h768'],
    'Static sparse': ['04_static_sparse_n16_v2'],
    'Dense':         ['04_dense_n16_v2'],
    'Block-sparse':  ['04_block_sparse_n16_v2'],
}

LR = 'optimizer.learning_rate'

# The swept axes of each arm. The learning rate is last everywhere; the arms
# with no other axis are tuned on it alone.
AXES = {
    'SET':           ['target_hidden_units', 'algorithm.evolve_frequency', LR],
    'DEEP-R':        ['target_hidden_units', 'algorithm.l1',
                      'algorithm.noise_ratio', LR],
    'Static sparse': ['target_hidden_units', LR],
    'Dense':         [LR],
    'Block-sparse':  [LR],
}

# How each axis reads on an x scale. The learning-rate grid is powers of two;
# l1 spans decades; the rest are plotted linearly -- widths include 768 and
# noise_ratio includes 0, neither of which survives a log axis.
AXIS_STYLE = {
    LR:                          dict(label='Step-size', log_base=2),
    'target_hidden_units':       dict(label='Hidden units', log_base=None),
    'algorithm.evolve_frequency':dict(label='Evolve frequency (steps)', log_base=None),
    'algorithm.l1':              dict(label=r'$\alpha$  (L1)', log_base=10),
    'algorithm.noise_ratio':     dict(label=r'Noise ratio  $\sqrt{2\eta T}/\eta\alpha$',
                                      log_base=None),
}

FIG_DIR = '../figures/generated'   # output, rewritten on every re-run

TAIL_FRAC = 0.10   # asymptotic = mean over the final 10% of logged periods

In [ ]:
# Run once to fetch from Comet, then leave commented out.
# from multi_mnist.analysis import download_project
# download_project(PROJECT, output_dir='data')

cfg_all, run_all = load_export(PROJECT, data_dir='data')
cfg_all = normalize_columns(cfg_all)
print(f'{len(cfg_all)} trials, {len(run_all)} metric rows')

## One row per grid cell

`asymptotic_metric` averages the final `TAIL_FRAC` of each trial's logged loss,
nulls any trial flagged diverged, and aggregates with `skipna=False` — so a
single bad seed makes the whole cell NaN rather than quietly averaging away.
NaNs are kept from here on. They drop out of the "best over the rest" step
because a NaN can never be the minimum, and a parameter value whose every
alternative is NaN stays NaN and leaves a visible gap in the curve.

In [ ]:
def cell_table(method):
    """Asymptotic loss for every cell of one arm's grid."""
    axes = AXES[method]
    cfg, run = filter_sweep(cfg_all, run_all, SWEEPS[method], cell_cols=axes)
    # asymptotic_metric always groups by the learning rate, so pass the others.
    others = [a for a in axes if a != LR]
    return asymptotic_metric(cfg, run, 'loss', others, tail_frac=TAIL_FRAC)


cells_by_method = {m: cell_table(m) for m in SWEEPS}

for m, tab in cells_by_method.items():
    n_nan = int(tab['_metric'].isna().sum())
    print(f'{m:14s} {len(tab):3d} cells, {n_nan:3d} NaN '
          f'(diverged or a NaN seed), best loss {tab["_metric"].min():.3f}')

## Sensitivity: best-of-the-rest at each value

For each value of the axis, the minimum over every other hyperparameter.

In [ ]:
def sensitivity(method, x_col):
    """Best (lowest) asymptotic loss achievable at each value of ``x_col``."""
    tab = cells_by_method[method]
    # min() skips NaN, so a diverged alternative does not hide a good one;
    # an all-NaN value yields NaN and breaks the line, which is the honest
    # rendering of "nothing at this setting finished".
    out = (tab.groupby(x_col, dropna=False)['_metric']
              .min().reset_index().sort_values(x_col))
    return out


def plot_method(method, fig_width=4.0, ylim=None):
    axes = AXES[method]
    fig, axs = plt.subplots(1, len(axes), figsize=(fig_width * len(axes), 3.4),
                            squeeze=False)
    for ax, col in zip(axs[0], axes):
        style = AXIS_STYLE[col]
        sub = sensitivity(method, col)
        plot_sensitivity(
            sub, ax=ax, x_col=col, y_col='_metric',
            xlabel=style['label'], ylabel='Asymptotic loss',
            log_base=style['log_base'], ylim=ylim)
        if not style['log_base']:
            # plot_sensitivity only pins ticks on a log axis; on a linear one
            # matplotlib would invent its own (300, 400, ...) and hide which
            # values were actually run.
            ticks = sorted(sub[col].dropna().unique())
            ax.set_xticks(ticks)
            ax.set_xticklabels([f'{t:g}' for t in ticks])
    fig.suptitle(method, y=1.02)
    fig.tight_layout()
    return fig, axs

### SET

Three axes: the width the budget is spread over, how often the
prune-and-regrow event fires (ζ is derived from it, holding the paper's
per-example rewiring rate fixed), and the step-size.

In [ ]:
fig, _ = plot_method('SET')
save_fig('04_sensitivity_set', fig_dir=FIG_DIR)

### DEEP-R

Four axes. `l1` is the strength of the pull toward zero that drives pruning,
and `noise_ratio` is the Langevin noise scaled to that pull — 0 turns the noise
off entirely, leaving deterministic sign-flip pruning.

In [ ]:
fig, _ = plot_method('DEEP-R')
save_fig('04_sensitivity_deep_r', fig_dir=FIG_DIR)

### Static sparse — SET's initialization, frozen

In [ ]:
fig, _ = plot_method('Static sparse')
save_fig('04_sensitivity_static_sparse', fig_dir=FIG_DIR)

### The two fixed-connectivity arms

Both are tuned on the step-size alone: block-sparse sits at the width that
defines the shared budget, and dense's width is derived from that budget
(16 units, since a fully connected unit costs 12,704 weights at 16 tasks).
Note the two use different step-size windows — dense diverges above 2⁻⁹.

In [ ]:
for m in ('Block-sparse', 'Dense'):
    fig, _ = plot_method(m, fig_width=4.6)
    save_fig(f'04_sensitivity_{m.lower().replace("-", "_")}', fig_dir=FIG_DIR)

## All arms on one step-size axis

The one axis every arm shares, so the five are directly comparable.

In [ ]:
from multi_mnist.analysis import get_color_palette

fig, ax = plt.subplots(figsize=(5.5, 3.8))
# A class->color mapping, so SET, DEEP-R, dense and block-sparse keep the
# colors they carry in the rest of the paper.
colors = get_color_palette(list(SWEEPS))
for method in SWEEPS:
    sub = sensitivity(method, LR)
    ax.plot(sub[LR], sub['_metric'], '-o', color=colors[method], label=method)
ax.set_xscale('log', base=2)
ticks = sorted(set().union(*[set(sensitivity(m, LR)[LR].dropna()) for m in SWEEPS]))
ax.set_xticks(ticks)
ax.set_xticklabels([f'$2^{{{int(np.log2(v))}}}$' for v in ticks])
ax.set_xlabel('Step-size')
ax.set_ylabel('Asymptotic loss')
ax.grid(True, alpha=0.4)
ax.legend(loc='best')
fig.tight_layout()
save_fig('04_sensitivity_all_step_size', fig_dir=FIG_DIR)

## Best cell per arm

The winning configuration behind each curve's minimum.

In [ ]:
rows = []
for method, axes in AXES.items():
    tab = cells_by_method[method]
    best = tab.loc[tab['_metric'].idxmin()]
    rows.append({'arm': method, 'loss': round(float(best['_metric']), 4),
                 **{a.split('.')[-1]: best[a] for a in axes}})
pd.DataFrame(rows).set_index('arm')